In [1]:
import geopandas as gpd
import pandas as pd

municipios = gpd.read_file('../temp/limites/SHP_ETRS89/recintos_municipales_inspire_peninbal_etrs89/recintos_municipales_inspire_peninbal_etrs89.shp')
municipio = municipios[(municipios.NAMEUNIT.str.startswith('Sangü')) | (municipios.NAMEUNIT.str.startswith('Aib'))]

municipio.to_crs(25830, inplace=True)

/home/nano/anaconda3/envs/jupyter/lib/python3.12/site-packages/geopandas/geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [2]:
incendios1 = gpd.read_file('BRUTOS/HISTORICO/FOREST_Pol_HcoIncendio.shp')
incendios2 = gpd.read_file('BRUTOS/HISTORICO/FOREST_Pol_HcoIncendioA.shp')
incendios_totales = pd.concat([incendios1, incendios2])
incendios_buffer = incendios_totales[incendios_totales.intersects(municipio.dissolve().iloc[0].geometry.buffer(30000))]
incendios = incendios_buffer.sort_values('SUPQUEMADA', ascending=False).iloc[:10]

incendios.loc[incendios.FECHA == '2016', 'FECHA'] = '25/08/2016'
incendios.loc[incendios.FECHA == '1994', 'FECHA'] = '15/07/1994'
incendios.loc[incendios.FECHA == '2014', 'FECHA'] = '18/07/2014'
incendios.loc[incendios.FECHA == '2009', 'FECHA'] = '22/07/2009'

In [3]:
peg = gpd.read_file('PEG/propuesta_PEG.gpkg')
peg['superficie'] = round((peg.geometry.area / 10_000), 2)

# TEXTOS

In [4]:
texto_historico = '''Según los datos de la base de datos de incendios históricos del IDENA (Infraestructura de Datos Espaciales de Navarra) \
no se ha registrado ningún incendio forestal dentro de los municipios de Sangüesa y Aibar en el periodo comprendido entre los años 1985 y 2023. \
Pero si establecemos un buffer de 30 kilómetros alrededor de los límites de ambos municipios, podemos observar que sí hay registrados múltiples incendios, \
muchos de ellos mayores de 500 ha que es límite que se establece para considerarlos GIF (Grandes Incendios Forestales). Se muestra un mapa con el perímetro de los incendios \
forestales registrados en Navarra, en rojo se muestran los perímetros de aquellos incendios que cuya distancia a los municipios de Sangüesa y Aibar \
es inferior a 30 kilómetros'''

In [5]:
texto_tabla_histórico = '''A continuación se muestra una tabla con los 10 incendios más grandes a menos de 30 kilómetros de los municipios de Sangüesa y Aibar. \
Se observa que todos ellos se produjeron en el periodo estival y que 7 de los incendios tienen una superficie de más de 500 hectáreas'''

In [6]:
texto_análisis_histórico = '''Con el fin de obtener un análisis de las condiciones meteorológicas que se dieron en estos incendios se ha creado una ficha de cada\
uno de ellos en la que se muestra los mapas de 500 y 850 hpa. De esta manera podemos hacernos una idea de las condiciones metereológicas más favorables para el \
desarrollo de grandes incendios forestales cerca de la zona de estudio'''

In [7]:
texto_flammap = "Se observa que la situación sinóptica recurrente en los incendios analizados es la adveción de masas de aire cálidas del sur, \
con la isoterma de 25\textdegree C a 850 hpa ocupando buena parte de la península, lo que se traduce en temperaturas altas, humedades relativas bajas y \
vientos de componente sur en superficie, con influencia de vientos ascendentes de ladera en zonas con relieve. \
Teniendo esto en cuenta se han hecho 4 simulaciones\
dentro del área de estudio usando el simulador Flammap. Como punto de origen del incendio se han escogido los puntos más probables de ignición y que pueden ser \
más favorables para el avance del incendio: fondos de barranco, zonas donde hay convergencia de vientos, zonas donde la zona agrícola se junta con la forestal...\
A continuación se muestran los 4 perímetros finales que se han obtenido para una simulación de avance de fuego de 6 horas"

In [8]:
texto_peg = "Del análisis de los perímetros obtenidos se han diseñado un total de 9 PEGs (Puntos Estratégicos de gestión), que son los siguientes:"

# SECCIONES

In [9]:
import os
from pylatex import Document, Command, Section, Subsection, Subsubsection, Figure, Tabular, NewPage, Center, StandAloneGraphic, Package, MiniPage
from pylatex.utils import NoEscape, italic

def maquetar_peg(doc, row): 
    # Nombres de las imágenes que debes tener en tu carpeta
    img1, img2, img3 = f'FINALES/PEG/peg_ubicacion_{row.id}.png', f'FINALES/PEG/peg_{row.id}.png', f'FINALES/PEG/peg_topo_{row.id}.png'
    
    with doc.create(Subsubsection(f'Información PEG {int(row.id)}')):
        
        # ----------------------------------------------------
        # FILA 1: Dos imágenes en paralelo
        # ----------------------------------------------------
        
        with doc.create(Figure(position='h!')) as f1:
            with doc.create(Center()):
                # Primera imagen (ocupa el 45% del ancho del texto)
                with f1.create(MiniPage(width=NoEscape(r'0.3\textwidth'))) as mp1:
                    if os.path.exists(img1):
                        mp1.append(StandAloneGraphic(image_options=NoEscape(r'width=\textwidth'), filename=img1))
                    else:
                        mp1.append(NoEscape(r'\centering\textbf{[Falta img1.png]}'))
                    mp1.append(NoEscape(r'\captionof*{figure}{Ubicación del PEG dentro del municipio}'))
                
                # Espacio horizontal entre las dos imágenes
                f1.append(NoEscape(r'\hfill'))
                
                # Bloque derecho para la Tabla (centrada verticalmente con [c])
                with f1.create(MiniPage(width=NoEscape(r'0.65\textwidth'), pos='c')) as mp_tabla:
                    mp_tabla.append(NoEscape(r'\centering'))
                    mp_tabla.append(NoEscape(r'\captionof*{table}{Datos}'))
                    mp_tabla.append(NoEscape(r'\vspace{0.5em}')) # Espacio entre el título y la tabla

                    mp_tabla.append(NoEscape(r'\renewcommand{\arraystretch}{1.4}'))
                    
                    # Estructura de la tabla (2 columnas con líneas)
                    with mp_tabla.create(Tabular(r'p{3.2cm}>{\raggedleft\arraybackslash}p{7cm}')) as table:
                        table.append(NoEscape(r'\toprule')) 
                        
                        # Cabecera de la tabla (opcional: puedes darle un fondo gris más oscuro)
                        mp_tabla.append(NoEscape(r'\rowcolor{gray!25}')) 
                        table.add_row(('Tipo', row.tipo))
                        table.add_hline()
                        table.add_row(('Superficie', f'{row.superficie} ha'))
                        table.add_hline()
                        mp_tabla.append(NoEscape(r'\rowcolor{gray!25}')) 
                        table.add_row(('Vegetación', row.vegetacion))
                        table.add_hline()
                        table.add_row(('Observaciones', row.observaciones))
                        table.append(NoEscape(r'\bottomrule'))

        # Añadir un espacio vertical de separación entre filas
        doc.append(NoEscape(r'\vspace{2em}'))
        
        # ----------------------------------------------------
        # FILA 2: Tercera imagen (izquierda) y Tabla (derecha)
        # ----------------------------------------------------
        
        with doc.create(Figure(position='h!')) as f2:
            with doc.create(Center()):
                # Bloque izquierdo para la Tercera Imagen
                # Segunda imagen (ocupa el 45% del ancho del texto)
                with f2.create(MiniPage(width=NoEscape(r'0.45\textwidth'))) as mp2:
                    if os.path.exists(img2):
                        mp2.append(StandAloneGraphic(image_options=NoEscape(r'width=\textwidth'), filename=img2))
                    else:
                        mp2.append(NoEscape(r'\centering\textbf{[Falta img2.png]}'))
                    mp2.append(NoEscape(r'\captionof*{figure}{PEG sobre ortofoto}'))
                
                # Espacio horizontal intermedio
                f2.append(NoEscape(r'\hfill'))

                with f2.create(MiniPage(width=NoEscape(r'0.45\textwidth'))) as mp_img3:
                    if os.path.exists(img3):
                        mp_img3.append(StandAloneGraphic(image_options=NoEscape(r'width=\textwidth'), filename=img3))
                    else:
                        mp_img3.append(NoEscape(r'\centering\textbf{[Falta img3.png]}'))
                    mp_img3.append(NoEscape(r'\captionof*{figure}{PEG sobre mapa topográfico}'))
    doc.append(NewPage())

In [10]:
import os
from pylatex import Document, Subsection, NewLine, Figure, Tabular, Center, StandAloneGraphic, Package, MiniPage
from pylatex.utils import NoEscape, italic

def maquetar_incendios(doc, row): 
    # Nombres de las imágenes que debes tener en tu carpeta
    try:
        dia, mes, year = row.FECHA.split('/')
    except:
        print(row.FECHAEXTIN, row.FECHA)
        dia = '01'
        mes = '01'
        year = '2022'
    fwett = f'{year}{mes}{dia}'
    print(fwett)
    img1 = f'BRUTOS/HISTORICO/MAPAS/mapa_{row.NUMPARTE}.png' 
    img2 = f'BRUTOS/HISTORICO/WETTERZENTRALE/isohipsas_500_{fwett}.png' 
    img3 = f'BRUTOS/HISTORICO/WETTERZENTRALE/isohipsas_850_{fwett}.png'
    
    with doc.create(Subsection(f'Información Incendio {row.MUNICIPIO}')):
        
        # ----------------------------------------------------
        # FILA 1: Dos imágenes en paralelo
        # ----------------------------------------------------
        
        with doc.create(Figure(position='h!')) as f1:
            with doc.create(Center()):
                # Primera imagen (ocupa el 45% del ancho del texto)
                with f1.create(MiniPage(width=NoEscape(r'0.45\textwidth'))) as mp1:
                    if os.path.exists(img1):
                        mp1.append(StandAloneGraphic(image_options=NoEscape(r'width=\textwidth,height=0.45\textheight,keepaspectratio'), filename=img1))
                    else:
                        print(img1)
                        mp1.append(NoEscape(r'\centering\textbf{[Falta img1.png]}'))
                    mp1.append(NoEscape(r'\captionof*{figure}{Perímetro final del incendio}'))
                
                # Espacio horizontal entre las dos imágenes
                f1.append(NoEscape(r'\hfill'))
                
                # Bloque derecho para la Tabla (centrada verticalmente con [c])
                with f1.create(MiniPage(width=NoEscape(r'0.45\textwidth'), pos='c')) as mp_tabla:
                    mp_tabla.append(NoEscape(r'\centering'))
                    mp_tabla.append(NoEscape(r'\captionof*{table}{Datos}'))
                    mp_tabla.append(NoEscape(r'\vspace{0.5em}')) # Espacio entre el título y la tabla

                    mp_tabla.append(NoEscape(r'\renewcommand{\arraystretch}{1.4}'))
                    
                    # Estructura de la tabla (2 columnas con líneas)
                    with mp_tabla.create(Tabular(r'p{3.2cm}>{\raggedleft\arraybackslash}p{3.2cm}')) as table:
                        table.append(NoEscape(r'\toprule')) 
                        
                        # Cabecera de la tabla (opcional: puedes darle un fondo gris más oscuro)
                        table.append(NoEscape(r'\rowcolor{gray!25}')) 
                        table.add_row(('Fecha', row.FECHA))
                        table.add_hline()
                        table.add_row(('Fecha Extinción', row.FECHAEXTIN))
                        table.add_hline()
                        mp_tabla.append(NoEscape(r'\rowcolor{gray!25}')) 
                        table.add_row(('Muncipio', row.MUNICIPIO))
                        table.add_hline()
                        table.add_row(('Superficie quemada', f'{row.SUPQUEMADA} ha'))
                        table.append(NoEscape(r'\bottomrule'))

        # Añadir un espacio vertical de separación entre filas
        doc.append(NoEscape(r'\vspace{2em}'))
        
        # ----------------------------------------------------
        # FILA 2: Tercera imagen (izquierda) y Tabla (derecha)
        # ----------------------------------------------------
        
        with doc.create(Figure(position='h!')) as f2:
            with doc.create(Center()):
                # Bloque izquierdo para la Tercera Imagen
                # Segunda imagen (ocupa el 45% del ancho del texto)
                with f2.create(MiniPage(width=NoEscape(r'0.45\textwidth'))) as mp2:
                    if os.path.exists(img2):
                        mp2.append(StandAloneGraphic(image_options=NoEscape(r'width=\textwidth'), filename=img2))
                    else:
                        mp2.append(NoEscape(r'\centering\textbf{[Falta img2.png]}'))
                    mp2.append(NoEscape(r'\captionof*{figure}{Mapa de 500 hpa del día del incendio}'))
                
                # Espacio horizontal intermedio
                f2.append(NoEscape(r'\hfill'))

                with f2.create(MiniPage(width=NoEscape(r'0.45\textwidth'))) as mp_img3:
                    if os.path.exists(img3):
                        mp_img3.append(StandAloneGraphic(image_options=NoEscape(r'width=\textwidth'), filename=img3))
                    else:
                        mp_img3.append(NoEscape(r'\centering\textbf{[Falta img3.png]}'))
                    mp_img3.append(NoEscape(r'\captionof*{figure}{Mapa de 850 hpa del día del incendio}'))
    doc.append(NewPage())

In [11]:
def maquetar_tabla_incendios(section, incendios):
    section.append(
        NoEscape(r"\vspace{0.5em}")
    )  # Espacio entre el título y la tabla

    # Creamos un entorno cerrado de centrado
    with section.create(Center()) as contenedor_centrado:

        contenedor_centrado.append(NoEscape(r"\renewcommand{\arraystretch}{1.4}"))

        # Estructura de la tabla (Creamos la tabla DENTRO del contenedor centrado)
        with contenedor_centrado.create(
            Tabular(r"p{3.2cm}p{3.2cm}>{\raggedleft\arraybackslash}p{3.2cm}")
        ) as table:
            table.append(NoEscape(r"\toprule"))

            # CORRECCIÓN: El color de la fila debe ir dentro de la 'table', no de la sección
            table.append(NoEscape(r"\rowcolor{gray!25}"))

            table.add_row(("Fecha", "Municipio", "Superficie Quemada"))
            table.add_hline()

            for i, incendio in incendios.iterrows():
                table.add_row(
                    (
                        incendio.FECHAEXTIN,
                        incendio.MUNICIPIO,
                        incendio.SUPQUEMADA,
                    )
                )

            table.append(NoEscape(r"\bottomrule"))

In [12]:
def historico(doc, incendios):
    with doc.create(Section('Incendios históricos')) as s1:
        s1.append(NoEscape(texto_historico))

        s1.append(NoEscape(r'\vspace{2em}'))
        with s1.create(Figure(position='H')) as historico_img:
            historico_img.add_image('FINALES/IMAGENES/analisis_historico.png')
            historico_img.add_caption('Incendios en la Comunidad Foral de Navarra. Fuente IDENA')

        s1.append(NoEscape(r'\vspace{2em}'))
        s1.append(NoEscape(r'\par ' + texto_tabla_histórico))
        s1.append(NewLine())
        maquetar_tabla_incendios(s1, incendios)

        s1.append(NoEscape(r'\vspace{2em}'))
        s1.append(NoEscape(r'\par ' + texto_análisis_histórico))
        s1.append(NewPage())
        for i, row in incendios.iterrows():
            maquetar_incendios(s1, row)

# REDACCIÓN

In [13]:
doc = Document(
    default_filepath='redaccion_peg',
    geometry_options={'margin': '2cm'}
)

# Añadir paquetes necesarios para subfiguras y manejo de espacios
doc.packages.append(Package('subcaption'))
doc.packages.append(Package('array'))
doc.packages.append(Package('float'))
doc.packages.append(Package('booktabs'))
doc.packages.append(Package('xcolor', options=['table'])) # 'table' permite colorear filas
doc.packages.add(Package("setspace"))

doc.append(Command("onehalfspacing"))

# Ocultar fecha y añadir título
doc.preamble.append(NoEscape(r'\date{}'))
doc.preamble.append(NoEscape(r'\title{Maquetación Avanzada en PyLaTeX}'))
#    doc.append(NoEscape(r'\maketitle'))

historico(doc, incendios)

with doc.create(Section('Diseño PEGs')) as s2:
    with s2.create(Subsection('Simulaciones')) as ss21:
        ss21.append(texto_flammap)
        for i in range(1, 5):
            with ss21.create(Subsubsection(f'Simulación {i}')) as sss211:
                with sss211.create(Figure(position='H')) as img:
                    img.add_image(f'FINALES/SIMULACIONES/simulacion_SUR_{i}.png')
                    img.add_caption(f'Simulación {i} con viento sur. Fuente: elaboración propia a partir de datos obtenidos con Flammap')

    with s2.create(Subsection('Puntos Estratégicos de Gestión')) as ss22:
        ss22.append(texto_peg)
        ss22.append(NewPage())
        for i, row in peg.iterrows():
            maquetar_peg(s2, row)

# 6. Compilar el archivo PDF
doc.generate_tex()

20220618
20220618
20160825
19940715
20140718
20230824
20090722
20220616
20220615
20220725
